# Modele Rekurencyjne

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !unzip pakiet.zip

In [ ]:
import pickle
import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from utils import pad_collate, train_composer_classifier, evaluate_accuracy
from model import LSTMComposerClassifier

In [ ]:
import importlib
import utils

importlib.reload(utils)
from utils import (
    pad_collate,
    test_pad_collate,
    train_composer_classifier,
    evaluate_accuracy
)

In [ ]:
from torch.utils.data import DataLoader, random_split

FOLDER_DIR = "pakiet/"

with open(FOLDER_DIR + "train.pkl", "rb") as f:
    train_data = pickle.load(f)

with open(FOLDER_DIR + "test_no_target.pkl", "rb") as f:
    test_data = pickle.load(f)

# Split train into train/validation
train_size = int(0.8 * len(train_data))
valid_size = len(train_data) - train_size

train_dataset, valid_dataset = random_split(
    train_data,
    [train_size, valid_size]
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=pad_collate
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=pad_collate
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False,
    collate_fn=test_pad_collate
)

In [ ]:
# Scan the loader to find the exact range of your raw chord data
absolute_min = 0
absolute_max = 0

for x_batch, _, _ in train_loader:
    b_min = x_batch.min().item()
    b_max = x_batch.max().item()
    if b_min < absolute_min: absolute_min = b_min
    if b_max > absolute_max: absolute_max = b_max

print(f"Raw data minimum index: {absolute_min}") # Probably -1
print(f"Raw data maximum index: {absolute_max}") # Probably 191


Raw data minimum index: 0
Raw data maximum index: 192


In [ ]:
x_padded, lengths, y = next(iter(train_loader))

print("x_padded shape:", x_padded.shape)
print("lengths shape:", lengths.shape)
print("labels shape:", y.shape)

print("x_padded:", x_padded)

vocab_size = int(x_padded.max() + 2)
print("vocab_size:", vocab_size)

x_padded shape: torch.Size([32, 1401])
lengths shape: torch.Size([32])
labels shape: torch.Size([32])
x_padded: tensor([[  1,  89,  93,  ...,   0,   0,   0],
        [145,  42,  13,  ...,   0,   0,   0],
        [  1,   2, 181,  ...,   0,   0,   0],
        ...,
        [145, 147,  51,  ...,   0,   0,   0],
        [  0,   0,   0,  ...,   0,   0,   0],
        [  0,   0,   0,  ...,   0,   0,   0]])
vocab_size: 194


In [ ]:
VOCAB_SIZE = vocab_size
EMBEDDING_DIM = 64
HIDDEN_SIZE = 128
NUM_LAYERS = 3
OUT_SIZE = 5

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = LSTMComposerClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS, OUT_SIZE).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
history_losses = []

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
train_composer_classifier(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    valid_loader=valid_loader,
    losses_list=history_losses,
    epochs=100,
    patience=6,
    save_path="best_composer_model.pt",
)


Saved best model -> best_composer_model.pt
[1/100] Train Loss: 0.1636 | Valid Loss: 0.2124 | Valid Acc: 94.22%
[2/100] Train Loss: 0.1586 | Valid Loss: 0.2140 | Valid Acc: 94.05%
[3/100] Train Loss: 0.1532 | Valid Loss: 0.2339 | Valid Acc: 94.05%
[4/100] Train Loss: 0.1777 | Valid Loss: 0.2167 | Valid Acc: 94.22%
[5/100] Train Loss: 0.1596 | Valid Loss: 0.2127 | Valid Acc: 94.22%
Saved best model -> best_composer_model.pt
[6/100] Train Loss: 0.1491 | Valid Loss: 0.2121 | Valid Acc: 94.22%
[7/100] Train Loss: 0.1508 | Valid Loss: 0.2132 | Valid Acc: 94.05%
Saved best model -> best_composer_model.pt
[8/100] Train Loss: 0.1419 | Valid Loss: 0.2105 | Valid Acc: 94.39%
[9/100] Train Loss: 0.1429 | Valid Loss: 0.2118 | Valid Acc: 94.22%
[10/100] Train Loss: 0.1387 | Valid Loss: 0.2180 | Valid Acc: 94.22%
[11/100] Train Loss: 0.1332 | Valid Loss: 0.2148 | Valid Acc: 94.05%
[12/100] Train Loss: 0.1356 | Valid Loss: 0.2147 | Valid Acc: 94.05%
[13/100] Train Loss: 0.1416 | Valid Loss: 0.2234 | V

In [116]:
model.load_state_dict(torch.load("best_composer_model.pt"))
model.eval()

LSTMComposerClassifier(
  (embedding): Embedding(194, 64, padding_idx=2)
  (lstm): LSTM(64, 128, num_layers=3)
  (fc): Linear(in_features=128, out_features=5, bias=True)
)

In [117]:
val_accuracy = evaluate_accuracy(model, valid_loader, device)
print(f"\nFinal Validate Accuracy: {val_accuracy:.2f}%")


Final Validate Accuracy: 94.39%


In [119]:
import pandas as pd

model.eval()

all_predictions = []

with torch.no_grad():

    for x_batch, lengths in test_loader:

        x_batch = x_batch.to(device)

        batch_size = x_batch.size(0)

        hidden = model.init_hidden(batch_size, device)

        outputs, _ = model(x_batch, hidden)

        preds = torch.argmax(outputs, dim=1)

        all_predictions.extend(preds.cpu().numpy())

pd.DataFrame(all_predictions).to_csv(
    "pred_94%.csv",
    header=False,
    index=False
)

print("pred.csv saved")

pred.csv saved
